In [1]:
import os

import numpy as np
import matplotlib.pyplot as plt

import torch

from mt_DIDC_config import LABEL2LABEL, PROPERTY_KEY, NEW_LABELS

from tx_GAN_train import CustomDatasetTexturizer

os.environ["CUDA_VISIBLE_DEVICES"] = "0"  

%load_ext autoreload
%autoreload 2


In [2]:

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Selected device: {device}")
print(f'Num available GPUs: ', torch.cuda.device_count())


p = torch.cuda.get_device_properties()
print(f"Device: {p.name} (Memory: {p.total_memory / 1e9:.2f} GB)")

Selected device: cuda
Num available GPUs:  1
Device: NVIDIA TITAN RTX (Memory: 25.19 GB)


In [3]:
DATA_DIR = "./DIDC_multiclass_coro_v2_prep_2"

In [4]:
prop_files = sorted([f for f in os.listdir(DATA_DIR) if f.endswith('props.npy')])
mask_files = sorted([f for f in os.listdir(DATA_DIR) if f.endswith('mask.npy')])
print(len(prop_files),len(mask_files))

458 458


In [5]:
n_pats = 5
pat_idxs = np.random.choice(len(prop_files), size=n_pats, replace=False)
pat_idxs.sort()
print(pat_idxs)

[  2  67  75 339 351]


In [6]:
props_vect = np.load(os.path.join(DATA_DIR, prop_files[pat_idxs[0]]))
mask_vect = np.load(os.path.join(DATA_DIR, mask_files[pat_idxs[0]]))

for idx in pat_idxs[1:]:
    print(f"Loading patient index: {idx}")
    props = np.load(os.path.join(DATA_DIR, prop_files[idx]))
    mask = np.load(os.path.join(DATA_DIR, mask_files[idx]))
    props_vect = np.concatenate([props_vect, props], axis=0) 
    mask_vect = np.concatenate([mask_vect, mask], axis=0)

props_vect.shape, mask_vect.shape

Loading patient index: 67
Loading patient index: 75
Loading patient index: 339
Loading patient index: 351


((1516, 3, 384, 384), (1516, 384, 384))

In [6]:
pd = props[:, 0]
t1 = props[:, 1]
t2 = props[:, 2]

In [7]:
mask.shape

(311, 384, 384)

In [8]:
class_indices = np.arange(len(NEW_LABELS))[:, None, None, None]  # (22, 1, 1, 1)
vol_mask_onehot = mask[None, ...] == class_indices  # (22, 311, 384, 384)

In [9]:
vol_mask_onehot.shape

(22, 311, 384, 384)

In [ ]:
props_perm =  np.permute_dims(props, (1, 0, 2, 3))
dist = []
for i in range(vol_mask_onehot.shape[0]):
    class_mask = vol_mask_onehot[i] 
    class_props = props_perm[:, class_mask] 
    dist.append(class_props) 
    print(class_props.shape)


(3, 321620)
(3, 18793588)
(3, 961161)
(3, 1602899)
(3, 186317)
(3, 3546063)
(3, 0)
(3, 0)
(3, 244364)
(3, 256900)
(3, 307783)
(3, 286694)
(3, 2591448)
(3, 6456433)
(3, 6917045)
(3, 1480036)
(3, 76360)
(3, 292428)
(3, 872731)
(3, 23556)
(3, 262455)
(3, 378935)
